In [ ]:
# M6.4 Trends: Subgroup vs. Overall (26/30 Pandas Concepts)

# A subgroup trend tells you how one segment is performing
# The overall trend tells you how everything is performing together
# Great analyses are often result from comparing the two
# EX: Is the warehouse outperforming the company average?
# EX: Is a suppliers lead time improving faster than the network average?

# import libraries
import pandas as pd
import numpy as np

# create the dataframe

df = pd.DataFrame({
    "month":    ["Jan","Jan","Jan","Feb","Feb","Feb",
                 "Mar","Mar","Mar","Apr","Apr","Apr",
                 "May","May","May","Jun","Jun","Jun"],
    "supplier": ["Acme","GlobalCo","FastParts"] * 6,
    "spend":    [1500, 2200, 800, 1800, 1900, 950,
                 1200, 2400, 1100, 2100, 2000, 750,
                 1700, 2300, 900, 2000, 2100, 1050],
    "on_time":  [8, 9, 7, 9, 8, 8, 7, 10, 9, 10, 9, 7,
                 9, 10, 8, 10, 9, 9],
    "deliveries":[10, 10, 10, 10, 10, 10, 10, 10, 10,
                  10, 10, 10, 10, 10, 10, 10, 10, 10]
})

print(df)

   month   supplier  spend  on_time  deliveries
0    Jan       Acme   1500        8          10
1    Jan   GlobalCo   2200        9          10
2    Jan  FastParts    800        7          10
3    Feb       Acme   1800        9          10
4    Feb   GlobalCo   1900        8          10
5    Feb  FastParts    950        8          10
6    Mar       Acme   1200        7          10
7    Mar   GlobalCo   2400       10          10
8    Mar  FastParts   1100        9          10
9    Apr       Acme   2100       10          10
10   Apr   GlobalCo   2000        9          10
11   Apr  FastParts    750        7          10
12   May       Acme   1700        9          10
13   May   GlobalCo   2300       10          10
14   May  FastParts    900        8          10
15   Jun       Acme   2000       10          10
16   Jun   GlobalCo   2100        9          10
17   Jun  FastParts   1050        9          10


In [10]:
# Exercise 2: Overall Trend vs. Subgroup Trend
# Add three columns using .assign()

# overall_avg_spend — the mean spend across ALL rows using .transform("mean") on the full column
# supplier_avg_spend — the mean spend per supplier using groupby + transform
# vs_overall — each row's spend minus overall_avg_spend

df = (
    df
    .assign(
        overall_avg_spend = lambda x: x["spend"].mean(),
        supplier_avg_spend = lambda x: x.groupby(by="supplier")["spend"].transform("mean"),
        vs_overall = lambda x: x["spend"] - x["overall_avg_spend"]
)
)

print(df[["month", "supplier", "spend", "overall_avg_spend", "supplier_avg_spend", "vs_overall"]])

# filter for suppliers who have an average spend above the overall average
print(df[df["supplier_avg_spend"] > df["overall_avg_spend"]])


   month   supplier  spend  overall_avg_spend  supplier_avg_spend  vs_overall
0    Jan       Acme   1500        1597.222222         1716.666667  -97.222222
1    Jan   GlobalCo   2200        1597.222222         2150.000000  602.777778
2    Jan  FastParts    800        1597.222222          925.000000 -797.222222
3    Feb       Acme   1800        1597.222222         1716.666667  202.777778
4    Feb   GlobalCo   1900        1597.222222         2150.000000  302.777778
5    Feb  FastParts    950        1597.222222          925.000000 -647.222222
6    Mar       Acme   1200        1597.222222         1716.666667 -397.222222
7    Mar   GlobalCo   2400        1597.222222         2150.000000  802.777778
8    Mar  FastParts   1100        1597.222222          925.000000 -497.222222
9    Apr       Acme   2100        1597.222222         1716.666667  502.777778
10   Apr   GlobalCo   2000        1597.222222         2150.000000  402.777778
11   Apr  FastParts    750        1597.222222          925.00000

In [14]:
# Exercise 3 -- OTD Rate Subgroup vs. Network

# Add these columns:

# otd_rate — on-time delivery rate per row: on_time / deliveries * 100
# supplier_avg_otd — average OTD rate per supplier using groupby + transform
# network_avg_otd — average OTD rate across all suppliers using transform("mean")
# vs_network — supplier average OTD minus network average — positive means above network benchmark

df = (
    df
    .assign(
        otd_rate = lambda x: (x["on_time"] / x["deliveries"]) * 100,
        supplier_avg_otd = lambda x: x.groupby(by="supplier")["otd_rate"].transform("mean"),
        network_avg_otd = lambda x: x["otd_rate"].mean(),
        vs_network = lambda x : x["supplier_avg_otd"] - x["network_avg_otd"]
    )
)
print(df[["month", "supplier", "otd_rate", "supplier_avg_otd", "network_avg_otd", "vs_network"]])

# Then filter suppliers performing above the network average.
print(df[df["supplier_avg_otd"] > df["network_avg_otd"]])


   month   supplier  otd_rate  supplier_avg_otd  network_avg_otd  vs_network
0    Jan       Acme      80.0         88.333333        86.666667    1.666667
1    Jan   GlobalCo      90.0         91.666667        86.666667    5.000000
2    Jan  FastParts      70.0         80.000000        86.666667   -6.666667
3    Feb       Acme      90.0         88.333333        86.666667    1.666667
4    Feb   GlobalCo      80.0         91.666667        86.666667    5.000000
5    Feb  FastParts      80.0         80.000000        86.666667   -6.666667
6    Mar       Acme      70.0         88.333333        86.666667    1.666667
7    Mar   GlobalCo     100.0         91.666667        86.666667    5.000000
8    Mar  FastParts      90.0         80.000000        86.666667   -6.666667
9    Apr       Acme     100.0         88.333333        86.666667    1.666667
10   Apr   GlobalCo      90.0         91.666667        86.666667    5.000000
11   Apr  FastParts      70.0         80.000000        86.666667   -6.666667

In [18]:
# Exercise 4: MoM Spend Trend per Supplier
# Use .groupby() and .shift()
df = (
    df
    .assign(
        spend_lag1 = lambda x: x.groupby(by="supplier")["spend"].shift(periods=1),
        spend_mom_change = lambda x: x["spend"] - x["spend_lag1"],
        spend_mom_pct = lambda x: (x["spend_mom_change"] / x["spend_lag1"] * 100).round(1)
    )
)
print(df)
# Then filter rows where month-over-month spend growth exceeded 10%. 
# Answer in a comment: why does combining groupby with shift matter here vs a simple shift?
# groupby + shift resets the lag at each supplier boundary
# as the calculation rolls down the column you want it to reset when the supplier changes
print(df[df["spend_mom_pct"] > 10])

   month   supplier  spend  on_time  deliveries  overall_avg_spend  \
0    Jan       Acme   1500        8          10        1597.222222   
1    Jan   GlobalCo   2200        9          10        1597.222222   
2    Jan  FastParts    800        7          10        1597.222222   
3    Feb       Acme   1800        9          10        1597.222222   
4    Feb   GlobalCo   1900        8          10        1597.222222   
5    Feb  FastParts    950        8          10        1597.222222   
6    Mar       Acme   1200        7          10        1597.222222   
7    Mar   GlobalCo   2400       10          10        1597.222222   
8    Mar  FastParts   1100        9          10        1597.222222   
9    Apr       Acme   2100       10          10        1597.222222   
10   Apr   GlobalCo   2000        9          10        1597.222222   
11   Apr  FastParts    750        7          10        1597.222222   
12   May       Acme   1700        9          10        1597.222222   
13   May   GlobalCo 